# Lucas-Kanade sobre un vídeo (versión sencilla)

Notebook basado en el paper de Lucas y Kanade (1981). No hago el algoritmo completo, solo la parte de **resolver la ecuación** en una ventana alrededor de cada punto.

La idea del paper en corto:

- $F$ es la imagen nueva y $G$ la anterior. Buscamos el desplazamiento $h$ que cumple $G(x) = F(x + h)$.
- Si $h$ es pequeño: $F(x+h) \approx F(x) + h \, \partial F / \partial x$.
- Minimizando el error L2 en una ventana $R$ queda:

$$h = \left( \sum_{x \in R} \nabla F \, \nabla F^T \right)^{-1} \sum_{x \in R} \nabla F \,[G(x) - F(x)]$$

En 2D esto es un sistema de 2x2 con cinco sumas (las que menciona el paper): $\sum F_x^2$, $\sum F_y^2$, $\sum F_x F_y$, $\sum (G-F)F_x$ y $\sum (G-F)F_y$.

Pasos:
1. Poner la ruta del vídeo y leer dos frames seguidos
2. Pasar a gris y suavizar
3. Calcular las derivadas
4. Resolver la ecuación en un punto
5. Hacerlo en varios puntos y dibujar las flechas
6. Aplicarlo a todo el vídeo

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## 1. Ruta del vídeo y lectura de dos frames

Aquí hay que poner la ruta del vídeo. Si se pone `0` se usa la webcam.

In [ ]:
# >>>>>>>>>>  PONER AQUÍ LA RUTA DEL VÍDEO  <<<<<<<<<<
RUTA_VIDEO = "ruta/de/tu/video.mp4"     # ejemplo: "people2.mp4"  (o 0 para la webcam)

cap = cv2.VideoCapture(RUTA_VIDEO)
ok1, frame_ant = cap.read()     # frame anterior  -> G
ok2, frame_nue = cap.read()     # frame nuevo     -> F
cap.release()

if not (ok1 and ok2):
    print("No se han podido leer dos frames, revisa la ruta del vídeo")
else:
    print("Tamaño del frame:", frame_ant.shape)

## 2. Escala de grises y suavizado

El paper dice que la aproximación lineal solo vale si $h$ es pequeño, y que suavizar la imagen ayuda a ampliar ese rango (sección 4.3). Por eso paso a gris y aplico un desenfoque gaussiano suave.

In [ ]:
def preparar(frame):
    gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    gris = cv2.GaussianBlur(gris, (5, 5), 0)
    return gris.astype(np.float32)      # float para que las restas no den problemas

G = preparar(frame_ant)    # imagen anterior
F = preparar(frame_nue)    # imagen nueva

## 3. Derivadas

El paper (sección 4.4) estima la derivada con una diferencia entre píxeles vecinos, con $\Delta x$ de un píxel:

$$F'(x) \approx \frac{F(x+\Delta x) - F(x)}{\Delta x}$$

Lo hago igual en x e y sobre la imagen nueva $F$. También calculo la diferencia $G - F$ que aparece en la ecuación.

In [ ]:
def derivadas(F, G):
    Fx = np.zeros_like(F)
    Fy = np.zeros_like(F)
    Fx[:, :-1] = F[:, 1:] - F[:, :-1]      # derivada en x (columna siguiente - actual)
    Fy[:-1, :] = F[1:, :] - F[:-1, :]      # derivada en y (fila siguiente - actual)
    dif = G - F                            # G - F
    return Fx, Fy, dif

Fx, Fy, dif = derivadas(F, G)

## 4. Resolver la ecuación en un punto

Cojo una ventana cuadrada alrededor del punto $(x, y)$, hago las cinco sumas y resuelvo el sistema 2x2 $A\,h = b$:

$$A = \begin{pmatrix} \sum F_x^2 & \sum F_x F_y \\ \sum F_x F_y & \sum F_y^2 \end{pmatrix} \qquad b = \begin{pmatrix} \sum (G-F)F_x \\ \sum (G-F)F_y \end{pmatrix}$$

Si la ventana es una zona lisa (sin textura) o solo hay un borde, $A$ no se puede invertir bien (el paper comenta que solo se divide por cero si el gradiente es cero en todas partes). En ese caso no devuelvo nada.

In [ ]:
R = 7                    # radio de la ventana -> ventana de 15x15
UMBRAL_DET = 1e4         # si el determinante es menor, no nos fiamos (valor a ojo, se puede cambiar)
H_MAX = 20               # si h sale más grande que esto (en píxeles) tampoco nos fiamos

def lk_punto(Fx, Fy, dif, x, y, r=R):
    # ventana alrededor del punto
    wx = Fx[y-r:y+r+1, x-r:x+r+1]
    wy = Fy[y-r:y+r+1, x-r:x+r+1]
    wd = dif[y-r:y+r+1, x-r:x+r+1]

    # las cinco sumas del paper
    sxx = np.sum(wx * wx)
    syy = np.sum(wy * wy)
    sxy = np.sum(wx * wy)
    sxd = np.sum(wd * wx)
    syd = np.sum(wd * wy)

    A = np.array([[sxx, sxy],
                  [sxy, syy]])
    b = np.array([sxd, syd])

    if np.linalg.det(A) < UMBRAL_DET:
        return None

    h = np.linalg.solve(A, b)      # h = A^-1 b
    if np.linalg.norm(h) > H_MAX:
        return None
    return h                       # (hx, hy)

In [ ]:
# Probamos con un punto. Cambiar x, y por un punto que tenga textura (una esquina, un objeto...)
x, y = 200, 150

h = lk_punto(Fx, Fy, dif, x, y)
print("h =", h)

## 5. Varios puntos y flechas

Repito lo mismo en una rejilla de puntos y dibujo una flecha por cada uno. Multiplico $h$ por una escala para que las flechas se vean (los desplazamientos entre dos frames suelen ser de un píxel o menos).

In [ ]:
PASO = 20        # separación entre puntos de la rejilla
ESCALA = 5       # para ver mejor las flechas

def dibujar_flechas(frame, Fx, Fy, dif):
    salida = frame.copy()
    alto, ancho = Fx.shape
    for y in range(R, alto - R, PASO):
        for x in range(R, ancho - R, PASO):
            h = lk_punto(Fx, Fy, dif, x, y)
            if h is None:
                continue
            fin = (int(x + ESCALA * h[0]), int(y + ESCALA * h[1]))
            cv2.arrowedLine(salida, (x, y), fin, (0, 255, 0), 1)
    return salida

resultado = dibujar_flechas(frame_nue, Fx, Fy, dif)

plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(resultado, cv2.COLOR_BGR2RGB))
plt.title("Lucas-Kanade (una iteración) entre dos frames")
plt.axis("off")
plt.show()

## 6. Todo el vídeo

Lo mismo pero frame a frame. Se para con Esc. Vuelvo a poner la ruta por si se quiere probar con otro vídeo.

In [ ]:
# >>>>>>>>>>  PONER AQUÍ LA RUTA DEL VÍDEO (otra vez)  <<<<<<<<<<
RUTA_VIDEO = "ruta/de/tu/video.mp4"     # o 0 para la webcam

cap = cv2.VideoCapture(RUTA_VIDEO)
ok, frame_ant = cap.read()
G = preparar(frame_ant)

while ok:
    ok, frame = cap.read()
    if not ok:               # se acabó el vídeo
        break

    F = preparar(frame)
    Fx, Fy, dif = derivadas(F, G)
    salida = dibujar_flechas(frame, Fx, Fy, dif)

    cv2.imshow('Lucas-Kanade', salida)

    G = F                    # el frame nuevo pasa a ser el anterior
    if cv2.waitKey(1) == 27: # Esc para salir
        break

cap.release()
cv2.destroyAllWindows()

## Qué no hemos hecho (y el paper sí)

- **No iteramos.** Solo hacemos un paso. El paper mueve $F$ según el $h$ obtenido y repite (ecuación 10).
- **No ponderamos** con $w(x)$ (ecuación 5).
- **No usamos pirámides** (coarse-fine). Por eso solo funciona con movimientos pequeños entre frames.
- **No incluimos rotación, escala ni brillo/contraste** (sección 4.6).

Si el movimiento es grande, las flechas salen mal o no salen, y una forma fácil de probar es saltarse frames o usar un vídeo con menos movimiento.